In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/yvvswaraag@gmail.com/capgemini-data-engineering-training/FinalProject/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","bankingdataplatform","Catalog")
dbutils.widgets.text("data_source","batch","Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://swaraag-finalpro/Bankingdata/{data_source}"
print(base_path)


In [0]:
# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.{data_source}"

In [0]:
df = spark.read.options(header = True,inferSchema = True).csv(F"{base_path}/*.csv").withColumn("read_timestamp",F.current_timestamp()).select("*","_metadata.file_name","_metadata.file_size")
print("Total Rows: ", df.count())
df.show(5)

In [0]:
df.write.format("delta")\
    .option("enableChandeDataFeed","true").mode("append")\
        .saveAsTable(bronze_table)

## SILVER

In [0]:
from pyspark.sql.functions import col, upper, trim, when, current_timestamp

print("🔄 Starting Silver Layer Transformations...\n")

# Read from Bronze table
print(f"📖 Reading from Bronze: {bronze_table}")
bronze_df = spark.table(bronze_table)

initial_count = bronze_df.count()
print(f"   Initial records: {initial_count:,}\n")

# ============================================
# TRANSFORMATION 1: Deduplication by txn_id
# ============================================
print("✅ Rule 1: Deduplication by txn_id")
duplicates_count = bronze_df.count() - bronze_df.select("txn_id").distinct().count()
print(f"   Duplicates found: {duplicates_count:,}")

# Keep first occurrence, drop duplicates
silver_df = bronze_df.dropDuplicates(["txn_id"])
print(f"   After dedup: {silver_df.count():,} records\n")

# ============================================
# TRANSFORMATION 2: Validation - amount > 0
# ============================================
print("✅ Rule 2: Validation - Remove negative/zero amounts")
invalid_amounts = silver_df.filter((col("amount") <= 0) | col("amount").isNull()).count()
print(f"   Invalid amounts found: {invalid_amounts:,}")

silver_df = silver_df.filter(col("amount") > 0)
print(f"   After validation: {silver_df.count():,} records\n")

# ============================================
# TRANSFORMATION 3: Filtering - status = SUCCESS
# ============================================
print("✅ Rule 3: Filtering - Keep only SUCCESS transactions")
failed_pending = silver_df.filter(col("status") != "SUCCESS").count()
print(f"   FAILED/PENDING transactions: {failed_pending:,}")

silver_df = silver_df.filter(col("status") == "SUCCESS")
print(f"   After filtering: {silver_df.count():,} records\n")

# ============================================
# TRANSFORMATION 4: Normalization - txn_type to DEBIT/CREDIT only
# ============================================
print("✅ Rule 4: Normalization - Standardize txn_type to DEBIT/CREDIT")
print("   Original txn_type values:")
silver_df.groupBy("txn_type").count().orderBy("txn_type").show(20, truncate=False)

# Normalize txn_type: Map all variations to DEBIT or CREDIT
silver_df = silver_df.withColumn(
    "txn_type",
    when(
        upper(trim(col("txn_type"))).isin(["DEBIT", "DEBIT", "DR", "D", "DB"]),
        "DEBIT"
    ).when(
        upper(trim(col("txn_type"))).isin(["CREDIT", "CREDIT", "CR", "C"]),
        "CREDIT"
    ).otherwise("UNKNOWN")  # For any unexpected values
)

# Remove any UNKNOWN values (data quality)
silver_df = silver_df.filter(col("txn_type") != "UNKNOWN")

print("   After normalization:")
silver_df.groupBy("txn_type").count().orderBy("txn_type").show()

# ============================================
# Additional Data Quality: Handle NULLs
# ============================================
print("✅ Additional: Remove records with NULL critical fields")
null_check = silver_df.filter(
    col("account_id").isNull() | 
    col("customer_id").isNull() | 
    col("currency").isNull() |
    col("channel").isNull()
).count()
print(f"   Records with NULLs in critical fields: {null_check:,}")

silver_df = silver_df.filter(
    col("account_id").isNotNull() & 
    col("customer_id").isNotNull() & 
    col("currency").isNotNull() &
    col("channel").isNotNull()
)

print(f"   After NULL removal: {silver_df.count():,} records\n")

# ============================================
# Add Processing Metadata
# ============================================
silver_df = silver_df.withColumn("_processed_ts", current_timestamp())

# ============================================
# Final Summary
# ============================================
final_count = silver_df.count()
removed_count = initial_count - final_count
removal_pct = (removed_count / initial_count * 100) if initial_count > 0 else 0

print("="*70)
print("📊 SILVER LAYER TRANSFORMATION SUMMARY")
print("="*70)
print(f"Initial Bronze records:        {initial_count:,}")
print(f"Final Silver records:          {final_count:,}")
print(f"Records removed:               {removed_count:,} ({removal_pct:.1f}%)")
print(f"\nData Quality Issues Cleaned:")
print(f"  - Duplicates:                {duplicates_count:,}")
print(f"  - Invalid amounts:           {invalid_amounts:,}")
print(f"  - Failed/Pending:            {failed_pending:,}")
print(f"  - NULL values:               {null_check:,}")
print("="*70)

# Preview cleaned data
print("\n📋 Sample Silver Data (First 10 rows):")
display(silver_df.select(
    "txn_id", "account_id", "txn_time", "txn_type", 
    "amount", "currency", "channel", "status"
).limit(10))

In [0]:
# Write cleaned data to Silver table
print(f"\n💾 Writing {final_count:,} records to Silver: {silver_table}")

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table)

print("✅ Successfully written to Silver layer!")

# Verify the write
print(f"\n🔍 Verifying Silver table:")
verify_df = spark.sql(f"SELECT COUNT(*) as total_records FROM {silver_table}")
verify_df.show()

# Show Silver table schema
print(f"\n📊 Silver Table Schema:")
spark.sql(f"DESCRIBE {silver_table}").show(truncate=False)

In [0]:
# Write all cleaned Silver data to Gold base table
print(f"\n💾 Writing cleaned data to Gold base table: {gold_table}")
print(f"   This creates a foundation for all Gold layer aggregations\n")

# Read from Silver (or use silver_df if still in memory)
silver_data = spark.table(silver_table)

print(f"📊 Records to write: {silver_data.count():,}")

# Write to Gold base table
silver_data.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(gold_table)

print(f"✅ Successfully written to Gold: {gold_table}")
print("   Now you can perform any aggregations on this base table!\n")

# Verify
verify_count = spark.table(gold_table).count()
print(f"🔍 Verification: {verify_count:,} records in {gold_table}")

## GOLD LAYER - ANALYTICS & KPIs

### Business Metrics & Fraud Detection

In [0]:
from pyspark.sql.functions import sum, count, max as spark_max, when

print("🏆 Computing Account Balances...\n")

# Read from Silver table
silver_data = spark.table(silver_table)

# Calculate account balances
account_balances = silver_data.groupBy("account_id", "customer_id").agg(
    # Calculate balance: CREDIT adds, DEBIT subtracts
    sum(
        when(col("txn_type") == "CREDIT", col("amount"))
        .otherwise(-col("amount"))
    ).alias("current_balance"),
    
    # Total credits
    sum(
        when(col("txn_type") == "CREDIT", col("amount"))
        .otherwise(0)
    ).alias("total_credits"),
    
    # Total debits
    sum(
        when(col("txn_type") == "DEBIT", col("amount"))
        .otherwise(0)
    ).alias("total_debits"),
    
    # Transaction count
    count("*").alias("txn_count"),
    
    # Last transaction time
    spark_max("txn_time").alias("last_txn_time")
).withColumn("last_updated", current_timestamp())

print(f"✅ Calculated balances for {account_balances.count():,} accounts\n")

# Show sample balances
print("📊 Sample Account Balances:")
display(account_balances.orderBy(col("current_balance").desc()).limit(10))

# Write to Gold table
gold_balances_table = f"{catalog}.{gold_schema}.account_balances"

print(f"\n💾 Writing to Gold: {gold_balances_table}")
account_balances.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_balances_table)

print("✅ Account balances loaded to Gold layer!")

In [0]:
from pyspark.sql.functions import sum, count, max as spark_max, when, expr, lit
from pyspark.sql.window import Window

print("⚠️  Computing Fraud Indicators...\n")

# Read from Silver table
silver_data = spark.table(silver_table)

# Calculate fraud indicators
fraud_indicators = silver_data.groupBy("account_id", "customer_id").agg(
    # High-value transaction count (> 100K)
    sum(
        when(col("amount") > 100000, 1).otherwise(0)
    ).alias("high_value_txn_count"),
    
    # Total transaction count for velocity
    count("*").alias("total_txn_count"),
    
    # Maximum transaction amount
    spark_max("amount").alias("max_txn_amount"),
    
    # Count distinct channels (unusual if > 5)
    count(col("channel")).alias("channel_count"),
    
    # Last transaction time
    spark_max("txn_time").alias("last_txn_time")
)

# Calculate fraud score (0-100 scale)
fraud_indicators = fraud_indicators.withColumn(
    "fraud_score",
    (
        # High-value transactions contribute 40 points
        when(col("high_value_txn_count") > 0, col("high_value_txn_count") * 10).otherwise(0) +
        # High velocity contributes 30 points (>20 txns)
        when(col("total_txn_count") > 20, 30).when(col("total_txn_count") > 10, 15).otherwise(0) +
        # Unusual max amount contributes 30 points (>200K)
        when(col("max_txn_amount") > 200000, 30).when(col("max_txn_amount") > 100000, 15).otherwise(0)
    ).cast("decimal(5,2)")
)

# Cap fraud score at 100
fraud_indicators = fraud_indicators.withColumn(
    "fraud_score",
    when(col("fraud_score") > 100, 100).otherwise(col("fraud_score"))
)

# Add velocity placeholders
fraud_indicators = fraud_indicators \
    .withColumn("velocity_1h", lit(0)) \
    .withColumn("velocity_24h", col("total_txn_count")) \
    .withColumn("unusual_channel_flag", col("channel_count") > 5) \
    .withColumn("analysis_timestamp", current_timestamp())

# Select final columns
fraud_indicators = fraud_indicators.select(
    "account_id",
    "customer_id",
    "fraud_score",
    "high_value_txn_count",
    "velocity_1h",
    "velocity_24h",
    "unusual_channel_flag",
    "max_txn_amount",
    "last_txn_time",
    "analysis_timestamp"
)

print(f"✅ Calculated fraud indicators for {fraud_indicators.count():,} accounts\n")

# Show high-risk accounts
print("⚠️  HIGH-RISK ACCOUNTS (Fraud Score > 50):")
high_risk = fraud_indicators.filter(col("fraud_score") > 50).orderBy(col("fraud_score").desc())
display(high_risk.limit(10))

print(f"\nTotal high-risk accounts: {high_risk.count():,}")

# Fraud score distribution
print("\n📊 Fraud Score Distribution:")
fraud_indicators.selectExpr(
    "CASE "
    "WHEN fraud_score = 0 THEN 'No Risk (0)' "
    "WHEN fraud_score <= 25 THEN 'Low Risk (1-25)' "
    "WHEN fraud_score <= 50 THEN 'Medium Risk (26-50)' "
    "WHEN fraud_score <= 75 THEN 'High Risk (51-75)' "
    "ELSE 'Critical Risk (76-100)' END as risk_category"
).groupBy("risk_category").count().orderBy("risk_category").show()

# Write to Gold table
gold_fraud_table = f"{catalog}.{gold_schema}.fraud_indicators"

print(f"\n💾 Writing to Gold: {gold_fraud_table}")
fraud_indicators.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(gold_fraud_table)

print("✅ Fraud indicators loaded to Gold layer!")

## 🏁 PIPELINE EXECUTION COMPLETE

### Medallion Architecture Summary:
- 🥉 **Bronze**: Raw data ingested from S3
- 🥈 **Silver**: Cleaned & validated transactions  
- 🥇 **Gold**: Business KPIs and fraud detection

In [0]:
# Final Pipeline Summary
print("="*70)
print("🏆 BANKING DATA PLATFORM - PIPELINE SUMMARY")
print("="*70)
print(f"\nCatalog: {catalog}")
print(f"Data Source: {data_source}")
print(f"S3 Path: {base_path}")

print("\n" + "="*70)
print("📊 LAYER STATISTICS")
print("="*70)

# Bronze stats
bronze_count = spark.table(bronze_table).count()
print(f"\n🥉 BRONZE LAYER: {bronze_table}")
print(f"   Total Records: {bronze_count:,}")

# Silver stats
silver_count = spark.table(silver_table).count()
print(f"\n🥈 SILVER LAYER: {silver_table}")
print(f"   Total Records: {silver_count:,}")
print(f"   Records Cleaned: {bronze_count - silver_count:,} ({(bronze_count - silver_count)/bronze_count*100:.1f}%)")

# Gold stats
gold_accounts = spark.table(gold_balances_table).count()
gold_fraud = spark.table(gold_fraud_table).count()
print(f"\n🥇 GOLD LAYER:")
print(f"   Account Balances: {gold_accounts:,} accounts")
print(f"   Fraud Indicators: {gold_fraud:,} accounts")

# High-risk accounts
high_risk_count = spark.table(gold_fraud_table).filter(col("fraud_score") > 50).count()
print(f"   High-Risk Accounts: {high_risk_count:,}")

print("\n" + "="*70)
print("✅ PIPELINE EXECUTION SUCCESSFUL")
print("="*70)
print("\nAll layers processed successfully!")
print("Data is now ready for business analytics and reporting.\n")

In [0]:


# Read from Silver table
silver_data = spark.table(silver_table)

record_count = silver_data.count()
print(f"📊 Records to write: {record_count:,}")

# Write to Gold base table
silver_data.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(gold_table)

print(f"\n✅ Successfully written to Gold: {gold_table}")
print("   Now you can perform any aggregations on this base table!\n")

# Verify
verify_count = spark.table(gold_table).count()
print(f"🔍 Verification: {verify_count:,} records in {gold_table}")
print("\n" + "="*70)
print("✅ GOLD BASE TABLE READY FOR ANALYTICS")
print("="*70)

In [0]:
# ============================================
# Total Count Across All Medallion Layers
# ============================================

print("\n" + "="*70)
print("📊 MEDALLION ARCHITECTURE - LAYER COUNTS")
print("="*70 + "\n")

# Bronze Layer Count
bronze_count = spark.table(bronze_table).count()
print(f"🥉 BRONZE LAYER: {bronze_table}")
print(f"   Total Records: {bronze_count:,}\n")

# Silver Layer Count
silver_count = spark.table(silver_table).count()
print(f"🥈 SILVER LAYER: {silver_table}")
print(f"   Total Records: {silver_count:,}")
records_cleaned = bronze_count - silver_count
cleaning_pct = (records_cleaned / bronze_count * 100) if bronze_count > 0 else 0
print(f"   Records Cleaned: {records_cleaned:,} ({cleaning_pct:.1f}%)\n")

# Gold Layer Count
gold_count = spark.table(gold_table).count()
print(f"🥇 GOLD LAYER: {gold_table}")
print(f"   Total Records: {gold_count:,}\n")

# Summary Table
print("="*70)
print("SUMMARY TABLE")
print("="*70)
print(f"{'Layer':<15} {'Table':<50} {'Count':>15}")
print("-"*70)
print(f"{'Bronze':<15} {bronze_table:<50} {bronze_count:>15,}")
print(f"{'Silver':<15} {silver_table:<50} {silver_count:>15,}")
print(f"{'Gold':<15} {gold_table:<50} {gold_count:>15,}")
print("="*70)

# Data Quality Metrics
data_quality_pct = (silver_count / bronze_count * 100) if bronze_count > 0 else 0
print(f"\n✅ Data Quality: {data_quality_pct:.1f}% of records passed validation")
print(f"❌ Records Filtered Out: {records_cleaned:,} ({cleaning_pct:.1f}%)")
print("\n" + "="*70)